In [42]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
import pickle
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

In [22]:
data = pd.read_csv("Churn_Modelling.csv")
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [23]:
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

In [24]:
## Encode categorical variable

label_encode_gender = LabelEncoder()
data['Gender'] = label_encode_gender.fit_transform(data['Gender'])

In [25]:
ohe_geo = OneHotEncoder(handle_unknown="ignore")

geo_encoder = ohe_geo.fit_transform(data[['Geography']]).toarray()
geo_en_df = pd.DataFrame(geo_encoder, columns=ohe_geo.get_feature_names_out(['Geography']))

geo_en_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [26]:
data = pd.concat([data.drop('Geography', axis=1), geo_en_df], axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [27]:
X = data.drop('EstimatedSalary', axis=1)
Y = data['EstimatedSalary']

In [29]:

# load the encoder & scaler
with open('ohe_geo.pkl', 'rb') as file:
    ohe_geo=pickle.load(file)


with open('label_encode_gender.pkl', 'rb') as file:
    label_encode_gender=pickle.load(file)


with open('scaler.pkl', 'rb') as file:
    scaler=pickle.load(file)        

In [30]:
# Split in training and testing
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

# Scale the features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## ANN Regression Problem Statement

In [31]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [36]:
# Model building

model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1) # when no activation fn is taken, then default act. fn. is Linear Act. Fn.
])

model.compile(optimizer='adam', loss='mean_absolute_error', metrics='mae')
model.summary()

Model: "sequential_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_9 (Dense)             (None, 64)                832       
                                                                 
 dense_10 (Dense)            (None, 32)                2080      
                                                                 
 dense_11 (Dense)            (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [43]:
# Set up the Tensorboard

log_dir = "regressionlogs/fit/" + datetime.datetime.now().strftime("%Y%m%d - %H%M%S")
tensorflow_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [44]:
early_stopping_callback = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [45]:
history = model.fit(
    X_train, Y_train, validation_data = (X_test, Y_test), epochs = 100,
    callbacks = [tensorflow_callback, early_stopping_callback]
)

Epoch 1/100


250/250 [==============================] - 8s 15ms/step - loss: 100386.8828 - mae: 100386.8828 - val_loss: 98549.4766 - val_mae: 98549.4766
Epoch 2/100
250/250 [==============================] - 2s 10ms/step - loss: 99744.8125 - mae: 99744.8125 - val_loss: 97246.1562 - val_mae: 97246.1562
Epoch 3/100
250/250 [==============================] - 2s 10ms/step - loss: 97423.1172 - mae: 97423.1172 - val_loss: 93781.0469 - val_mae: 93781.0469
Epoch 4/100
250/250 [==============================] - 2s 10ms/step - loss: 92702.3672 - mae: 92702.3672 - val_loss: 87836.0000 - val_mae: 87836.0000
Epoch 5/100
250/250 [==============================] - 2s 9ms/step - loss: 85658.6406 - mae: 85658.6406 - val_loss: 79982.8516 - val_mae: 79982.8516
Epoch 6/100
250/250 [==============================] - 2s 10ms/step - loss: 77140.6797 - mae: 77140.6797 - val_loss: 71413.4062 - val_mae: 71413.4062
Epoch 7/100
250/250 [==============================] - 3s 10ms/step - loss: 68542.1406 - mae: 685

In [47]:
%load_ext tensorboard
%tensorboard --logdir regressionlogs/fit

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 12132), started 0:00:52 ago. (Use '!kill 12132' to kill it.)

In [48]:
test_loss, test_mae = model.evaluate(X_test, Y_test)
print(f'Test MAE: {test_mae}')

63/63 [==============================] - 0s 2ms/step - loss: 50413.4609 - mae: 50413.4609
Test MAE: 50413.4609375


In [49]:
model.save('regression_model.h5')

c:\Users\harsh\anaconda3\envs\v_env\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
